### 导出 onnx

In [ ]:
import yaml
import torch

from ultralytics.nn.modules.head import Detect
from ultralytics.nn.tasks import DetectionModel


default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", "r"), Loader=yaml.FullLoader)
model_cfg = yaml.load(open("cfgs/yolov12.yaml", "r"), Loader=yaml.FullLoader)
model_cfg["scale"] = "n"

model = DetectionModel(cfg=model_cfg, ch=3, verbose=False)
pth = torch.load("logs/20260525-174123/model_epoch_339_0.6336.pth", map_location="cpu")
model.load_state_dict(pth["model"])
# model.fuse(True)
model = model.eval()

for m in model.modules():
    if isinstance(m, Detect):
        m.dynamic = False  # 禁用动态grid
        m.export = True    # 启用导出模式
        m.format = 'onnx'


dummy_input = torch.randn(1, 3, 160, 160)
model_out = model(dummy_input)
print(len(model_out))
for item in model_out:
    print(item.shape)
torch.onnx.export(
    model,
    dummy_input,
    "onnx/yolov12_test.onnx",
    opset_version=12,
    do_constant_folding=True,
    input_names=["input1"],
    output_names=["output0", "output1"],
    dynamo=None,
    dynamic_axes=None,
)

# simplify ONNX model
import onnx
import onnxsim
model_onnx = onnx.load("onnx/yolov12_test.onnx")
model_onnx, check = onnxsim.simplify(model_onnx)
onnx.save(model_onnx, "onnx/yolov12_test.onnx")

from fvcore import nn as fvnn

params = fvnn.parameter_count_table(model)
print(params)

### 导出save model

In [1]:
import yaml
import torch

from ultralytics.nn.modules.head import Detect
from ultralytics.nn.tasks import DetectionModel


default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", "r"), Loader=yaml.FullLoader)
model_cfg = yaml.load(open("cfgs/yolov12.yaml", "r"), Loader=yaml.FullLoader)
model_cfg["scale"] = "n"

model = DetectionModel(cfg=model_cfg, ch=3, verbose=False)
pth = torch.load("logs/20260525-174123/model_epoch_339_0.6336.pth", map_location="cpu")
model.load_state_dict(pth["model"])
model = model.eval()

for m in model.modules():
    if isinstance(m, Detect):
        m.dynamic = False  # 禁用动态grid
        m.export = True    # 启用导出模式
        m.format = 'onnx'

dummy_input = torch.randn(1, 3, 160, 160)
model_out = model(dummy_input)
print(len(model_out))
for item in model_out:
    print(item.shape)
    
torch.onnx.export(
    model,
    dummy_input,
    "onnx/yolov12_160_sm.onnx",
    opset_version=12,
    do_constant_folding=True,
    input_names=["input1"],
    output_names=["output0", "output1"],
    dynamo=None,
    dynamic_axes=None,
)

# simplify ONNX model
import onnx
import onnxslim
model_onnx = onnx.load("onnx/yolov12_160_sm.onnx")
model_onnx = onnxslim.slim(model_onnx)
onnx.save(model_onnx, "onnx/yolov12_160_sm.onnx")

# export to TFLite with onnx2tf
import onnx2tf
import numpy as np
from glob import glob
from PIL import Image

tmp_file = "tmp_tflite_int8_calibration_images.npy"
print(f"Using custom int8 calibration dataset for onnx2tf from {tmp_file}")
data = glob("data/blueberry_cls_v2/train/images/*.jpg")[:200]
images = []
for img in data:
    images.append(np.asarray(Image.open(img).convert('RGB').resize((160, 160))))
images = np.stack(images, 0)
np.save(str(tmp_file), images.astype(np.float32))
np_data = [["input1", tmp_file, [[[[0., 0., 0.]]]], [[[[255., 255., 255.]]]]]]
            
keras_model = onnx2tf.convert(
    input_onnx_file_path="onnx/yolov12_160_sm.onnx",
    output_folder_path="onnx/save_model/",
    not_use_onnxsim=True,
    verbosity="error",  # note INT8-FP16 activation bug https://github.com/ultralytics/ultralytics/issues/15873
    output_integer_quantized_tflite=True,
    quant_type="per-channel",  # "per-tensor" (faster) or "per-channel" (slower but more accurate)
    custom_input_op_name_np_data_path=np_data,
    disable_group_convolution=True,  # for end-to-end model compatibility
    enable_batchmatmul_unfold=True,  # for end-to-end model compatibility
)

/root/autodl-tmp/envs/yolo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FlashAttention is not available on this device. Using scaled_dot_product_attention instead.
2
torch.Size([1, 4, 500])
torch.Size([1, 1, 500])


/tmp/ipykernel_3204848/188930644.py:29: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/root/autodl-tmp/hanyu_other/yolo_fruit_detect/ultralytics/nn/modules/head.py:111: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if self.format != "imx" and (self.dynamic or self.shape != shape):
E0000 00:00:1779872447.235331 3204848 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when

Using custom int8 calibration dataset for onnx2tf from tmp_tflite_int8_calibration_images.npy


I0000 00:00:1779872452.084752 3204848 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2934 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:39:00.0, compute capability: 8.6
I0000 00:00:1779872457.515415 3204848 cuda_dnn.cc:529] Loaded cuDNN version 91002


Saved artifact at 'onnx/save_model/'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 160, 160, 3), dtype=tf.float32, name='input1')
Output Type:
  List[TensorSpec(shape=(1, 4, 500), dtype=tf.float32, name=None), TensorSpec(shape=(1, 1, 500), dtype=tf.float32, name=None)]
Captures:
  140516302093856: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  140516302093504: TensorSpec(shape=(3, 3, 3, 16), dtype=tf.float32, name=None)
  140516302093680: TensorSpec(shape=(16,), dtype=tf.float32, name=None)
  140516300185744: TensorSpec(shape=(4, 2), dtype=tf.int32, name=None)
  140516300184160: TensorSpec(shape=(3, 3, 8, 16), dtype=tf.float32, name=None)
  140516300186624: TensorSpec(shape=(3, 3, 8, 16), dtype=tf.float32, name=None)
  140516300179408: TensorSpec(shape=(32,), dtype=tf.float32, name=None)
  140516301989392: TensorSpec(shape=(1, 1, 32, 32), dtype=tf.float32, name=None)
  140516301987104: TensorSpec(shape=(3

W0000 00:00:1779872470.712802 3204848 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779872470.712830 3204848 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1779872470.760595 3204848 mlir_graph_optimization_pass.cc:425] MLIR V1 optimization pass is not enabled
W0000 00:00:1779872480.127225 3204848 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779872480.127248 3204848 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
I0000 00:00:1779872481.414279 3204848 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1779872481.414523 3204848 single_machine.cc:374] Starting new session
I0000 00:00:1779872481.416872 3204848 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2934 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:39:00.0, compute capability: 8.6
W0000 00:00:1779872481.977173 3204848 tf_t

W0000 00:00:1779872485.161319 3204848 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779872485.161348 3204848 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1779872502.085844 3204848 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779872502.085867 3204848 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1779872519.115069 3204848 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779872519.115095 3204848 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
W0000 00:00:1779872550.544396 3204848 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1779872550.544418 3204848 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


In [2]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path="onnx/save_model/yolov12_160_sm_full_integer_quant.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input details:")
for k, v in input_details[0].items():
    print(f"  {k}: {v}")
print("Output details:")
print(len(output_details))
for output_detail in output_details:
    for k, v in output_detail.items():
        print(f"  {k}: {v}")


Input details:
  name: serving_default_input1:0
  index: 0
  shape: [  1 160 160   3]
  shape_signature: [  1 160 160   3]
  dtype: <class 'numpy.int8'>
  quantization: (0.003921568859368563, -128)
  quantization_parameters: {'scales': array([  0.0039216], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}
  sparsity_parameters: {}
Output details:
2
  name: PartitionedCall:1
  index: 505
  shape: [  1   1 500]
  shape_signature: [  1   1 500]
  dtype: <class 'numpy.int8'>
  quantization: (0.00390625, -128)
  quantization_parameters: {'scales': array([  0.0039062], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}
  sparsity_parameters: {}
  name: PartitionedCall:0
  index: 503
  shape: [  1   4 500]
  shape_signature: [  1   4 500]
  dtype: <class 'numpy.int8'>
  quantization: (1.6155468225479126, -121)
  quantization_parameters: {'scales': array([     1.6155], dtype=float32), 'zero_points': array([-121], dtype=int32)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


### 测试 onnx

In [ ]:
import onnx
import random
import numpy as np
import onnxruntime as ort
import glob

from utils import sigmoid, xywh2xyxy, nms
from PIL import Image
from matplotlib import pyplot as plt

session = ort.InferenceSession("onnx/model_epoch_600_saved_model/model_epoch_600.onnx")
inputs = session.get_inputs()
print("Input name:", inputs[0].name)
print("Input shape:", inputs[0].shape)
print("Input type:", inputs[0].type)

# path = "strawberry_cls/test/images/20250831_101627_jpg.rf.baa9958c0e183de4acc059f56bd3e0d3.jpg"
imgs = glob.glob("strawberry_cls/test/images/*.jpg")
path = imgs[random.randint(0, len(imgs) - 1)]
image = np.asarray(Image.open(path).convert("RGB")).copy()
image = image.astype(np.float32).transpose(2, 0, 1) / 255.0
image = np.expand_dims(image, axis=0)

outputs = session.run(None, {inputs[0].name: image})
anchors = outputs[0][0].transpose(1, 0)

# split
boxes, cls = anchors[:, :4], anchors[:, 4:]
boxes = xywh2xyxy(boxes)
conf = cls.max(axis=1)
cls_id = cls.argmax(axis=1)

# filter
mask = conf > 0.25
boxes = boxes[mask]
cls = cls[mask]
conf = conf[mask]
cls_id = cls_id[mask]

# nms
pred_idx = nms(boxes, conf, iou_threshold=0.45)



# cls2label = {0: "fullripe", 1: "semiripe", 2: "unripe"}
# plt.imshow(image[0].transpose(1, 2, 0))
# for i in pred_idx:
#     x_min, y_min, x_max, y_max = boxes[i]
#     plt.gca().add_patch(plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
#                                       edgecolor='red', facecolor='none', linewidth=2))
#     plt.text(x_min, y_min - 10, f"{cls2label[cls_id[i]]} {conf[i]:.2f}", color='red', fontsize=12)

In [ ]:
# python: 检查 onnx 输入和值信息并找出含 3200 的维度
import onnx
m = onnx.load("onnx/yolov12_simplified.onnx")
onnx.checker.check_model(m)
def dims_of(vi):
    dims=[]
    t=vi.type.tensor_type
    for d in t.shape.dim:
        if d.HasField("dim_value"):
            dims.append(d.dim_value)
        elif d.HasField("dim_param"):
            dims.append(d.dim_param)
        else:
            dims.append(None)
    return dims

for vi in list(m.graph.input)+list(m.graph.value_info)+list(m.graph.output):
    try:
        ds = dims_of(vi)
    except Exception:
        continue
    if 3200 in ds:
        print("Found 3200 in:", vi.name, ds)

print("Model inputs:")
for i in m.graph.input:
    print(i.name, dims_of(i))

### 量化数据

In [ ]:
import os
import glob
import numpy as np
from PIL import Image

image_paths = sorted(glob.glob("strawberry_cls/train/images/*.jpg"))
if not image_paths:
    raise RuntimeError("No images found in strawberry_cls/train/images")

images = []
for image_path in image_paths:
    image = Image.open(image_path).convert("RGB").resize((640, 640))
    image_array = np.asarray(image, dtype=np.float32)  # 0~255
    images.append(image_array)

train_images = np.stack(images, axis=0).astype(np.float32)  # NHWC
os.makedirs("onnx", exist_ok=True)
np.save("onnx/train_images.npy", train_images)

print("saved:", "onnx/train_images.npy")
print("shape:", train_images.shape)
print("dtype:", train_images.dtype)
print("min/max:", train_images.min(), train_images.max())